# Лабораторная работа 4

## Часть 1

In [3]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import urllib.request
import tarfile
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import umap

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
df = pd.read_csv("factoryReports.csv")
print(df.head())
print("\nРаспределение классов:")
print(df['Category'].value_counts())

                                 Description            Category
0                    Scanner reel misaligned   Hydraulic Failure
1       Items getting stuck in scanner spool  Mechanical Failure
2  Burnt capacitor detected in control board    Software Failure
3              Robot arm movement inaccurate                Leak
4       Calibration lost after system reboot                Leak

Распределение классов:
Category
Leak                  74
Calibration Issue     67
Hydraulic Failure     66
Mechanical Failure    66
Sensor Failure        59
Electrical Failure    58
Overheating           56
Software Failure      54
Name: count, dtype: int64


In [5]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

tokenized_texts = [tokenize(t) for t in df['Description'].astype(str)]

counter = Counter()
for toks in tokenized_texts:
    counter.update(toks)

PAD_IDX = 0
UNK_IDX = 1
vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
max_vocab_size = 800
min_freq = 2

for word, freq in counter.most_common():
    if freq < min_freq:
        continue
    if len(vocab) >= max_vocab_size:
        break
    vocab[word] = len(vocab)

vocab_size = len(vocab)
print("Размер словаря:", vocab_size)

def numericalize(tokens):
    return [vocab.get(t, UNK_IDX) for t in tokens]

numericalized = [numericalize(toks) for toks in tokenized_texts]

lengths = [len(toks) for toks in tokenized_texts]
MAX_LEN = int(np.percentile(lengths, 95))
print(f"Максимальная длина (95-й перцентиль): {MAX_LEN}")

def pad_seq(seq, max_len=MAX_LEN):
    if len(seq) < max_len:
        return seq + [PAD_IDX] * (max_len - len(seq))
    else:
        return seq[:max_len]

X = torch.tensor([pad_seq(seq) for seq in numericalized], dtype=torch.long)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Category'].astype(str))
y = torch.tensor(y, dtype=torch.long)
num_classes = len(label_encoder.classes_)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Размер словаря: 82
Максимальная длина (95-й перцентиль): 6
Train: torch.Size([400, 6]), Val: torch.Size([100, 6])


In [6]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_dataset = TextDataset(X_train, y_train)
val_dataset   = TextDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [7]:
class RegularizedLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 num_classes, pad_idx=0, dropout_p=0.5, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional   # ← добавить эту строку
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_p if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout_p)
        fc_in_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_in_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (h_n, c_n) = self.lstm(emb)
        if self.bidirectional:
            last_hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            last_hidden = h_n[-1]
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden)
        return logits

In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total

In [9]:
embed_dims = [100, 200]
hidden_dims = [128, 256]
dropouts = [0.3, 0.5]
weight_decays = [1e-4, 1e-5]
num_layers = 2
bidirectional = True

best_val_acc = 0.0
best_params = {}

for embed_dim in embed_dims:
    for hidden_dim in hidden_dims:
        for dropout_p in dropouts:
            for weight_decay in weight_decays:
                print(f"\nTesting embed_dim={embed_dim}, hidden_dim={hidden_dim}, dropout={dropout_p}, wd={weight_decay}")
                model = RegularizedLSTM(
                    vocab_size=vocab_size,
                    embed_dim=embed_dim,
                    hidden_dim=hidden_dim,
                    num_layers=num_layers,
                    num_classes=num_classes,
                    pad_idx=PAD_IDX,
                    dropout_p=dropout_p,
                    bidirectional=bidirectional
                ).to(device)

                optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
                criterion = nn.CrossEntropyLoss()

                patience = 5
                best_val = 0.0
                no_improve = 0
                for epoch in range(1, 31):
                    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
                    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
                    if val_acc > best_val:
                        best_val = val_acc
                        no_improve = 0
                    else:
                        no_improve += 1
                        if no_improve >= patience:
                            break
                print(f"Val accuracy: {best_val:.3f}")
                if best_val > best_val_acc:
                    best_val_acc = best_val
                    best_params = {
                        'embed_dim': embed_dim,
                        'hidden_dim': hidden_dim,
                        'dropout_p': dropout_p,
                        'weight_decay': weight_decay
                    }

print("\nЛучшие параметры:", best_params)
print("Лучшая точность на валидации:", best_val_acc)


Testing embed_dim=100, hidden_dim=128, dropout=0.3, wd=0.0001
Val accuracy: 0.140

Testing embed_dim=100, hidden_dim=128, dropout=0.3, wd=1e-05
Val accuracy: 0.130

Testing embed_dim=100, hidden_dim=128, dropout=0.5, wd=0.0001
Val accuracy: 0.110

Testing embed_dim=100, hidden_dim=128, dropout=0.5, wd=1e-05
Val accuracy: 0.130

Testing embed_dim=100, hidden_dim=256, dropout=0.3, wd=0.0001
Val accuracy: 0.120

Testing embed_dim=100, hidden_dim=256, dropout=0.3, wd=1e-05
Val accuracy: 0.100

Testing embed_dim=100, hidden_dim=256, dropout=0.5, wd=0.0001
Val accuracy: 0.130

Testing embed_dim=100, hidden_dim=256, dropout=0.5, wd=1e-05
Val accuracy: 0.120

Testing embed_dim=200, hidden_dim=128, dropout=0.3, wd=0.0001
Val accuracy: 0.120

Testing embed_dim=200, hidden_dim=128, dropout=0.3, wd=1e-05
Val accuracy: 0.120

Testing embed_dim=200, hidden_dim=128, dropout=0.5, wd=0.0001
Val accuracy: 0.150

Testing embed_dim=200, hidden_dim=128, dropout=0.5, wd=1e-05
Val accuracy: 0.150

Testing e

In [10]:
best_params = {'embed_dim': 200, 'hidden_dim': 128, 'dropout_p': 0.5, 'weight_decay': 0.0001}

X_full = torch.cat([X_train, X_val], dim=0)
y_full = torch.cat([y_train, y_val], dim=0)
full_dataset = TextDataset(X_full, y_full)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

final_model = RegularizedLSTM(
    vocab_size=vocab_size,
    embed_dim=best_params['embed_dim'],
    hidden_dim=best_params['hidden_dim'],
    num_layers=2,
    num_classes=num_classes,
    pad_idx=PAD_IDX,
    dropout_p=best_params['dropout_p'],
    bidirectional=True
).to(device)

optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=best_params['weight_decay'])
criterion = nn.CrossEntropyLoss()

print("Финальное обучение на всех данных (train+val):")
for epoch in range(1, 31):
    train_loss, train_acc = train_epoch(final_model, full_loader, optimizer, criterion, device)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f}")
    
torch.save(final_model.state_dict(), 'factory_final_model.pt')
print("Модель сохранена")

Финальное обучение на всех данных (train+val):
Epoch 01 | train_loss=2.0863 train_acc=0.130
Epoch 02 | train_loss=2.0522 train_acc=0.176
Epoch 03 | train_loss=2.0388 train_acc=0.206
Epoch 04 | train_loss=2.0232 train_acc=0.200
Epoch 05 | train_loss=2.0076 train_acc=0.198
Epoch 06 | train_loss=2.0056 train_acc=0.212
Epoch 07 | train_loss=1.9996 train_acc=0.208
Epoch 08 | train_loss=1.9975 train_acc=0.212
Epoch 09 | train_loss=1.9892 train_acc=0.216
Epoch 10 | train_loss=1.9954 train_acc=0.202
Epoch 11 | train_loss=1.9890 train_acc=0.214
Epoch 12 | train_loss=1.9872 train_acc=0.204
Epoch 13 | train_loss=1.9842 train_acc=0.212
Epoch 14 | train_loss=1.9943 train_acc=0.210
Epoch 15 | train_loss=1.9969 train_acc=0.192
Epoch 16 | train_loss=1.9908 train_acc=0.196
Epoch 17 | train_loss=1.9800 train_acc=0.218
Epoch 18 | train_loss=1.9909 train_acc=0.212
Epoch 19 | train_loss=1.9782 train_acc=0.216
Epoch 20 | train_loss=1.9721 train_acc=0.208
Epoch 21 | train_loss=1.9925 train_acc=0.216
Epoch 22

In [11]:
model = RegularizedLSTM(
    vocab_size=vocab_size,
    embed_dim=best_params['embed_dim'],
    hidden_dim=best_params['hidden_dim'],
    num_layers=2,
    num_classes=num_classes,
    pad_idx=PAD_IDX,
    dropout_p=best_params['dropout_p'],
    bidirectional=True
).to(device)
model.load_state_dict(torch.load('factory_final_model.pt'))

val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
print(f"Итоговая точность на валидации: {val_acc:.3f}")

Итоговая точность на валидации: 0.150


## Часть 2

In [13]:
if not os.path.exists("aclImdb_v1.tar.gz"):
    print("Скачивание aclImdb_v1.tar.gz...")
    urllib.request.urlretrieve("https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz", "aclImdb_v1.tar.gz")
    with tarfile.open("aclImdb_v1.tar.gz", "r:gz") as tar:
        tar.extractall()
    print("Распаковано.")
else:
    print("Файл уже есть, распаковка не требуется.")

Файл уже есть, распаковка не требуется.


In [14]:
def preprocess(text):
    return text.lower()

def get_dataset(split='train', base_dir='aclImdb'):
    path_data = os.path.join(base_dir, split)
    path_neg = os.path.join(path_data, 'neg')
    path_pos = os.path.join(path_data, 'pos')
    neg_files = sorted(os.listdir(path_neg))
    pos_files = sorted(os.listdir(path_pos))
    X = []
    y = [0]*len(neg_files) + [1]*len(pos_files)
    for fname in neg_files:
        with open(os.path.join(path_neg, fname), 'r', encoding='utf-8') as f:
            X.append(preprocess(f.readline()))
    for fname in pos_files:
        with open(os.path.join(path_pos, fname), 'r', encoding='utf-8') as f:
            X.append(preprocess(f.readline()))
    return X, y

X_train_imdb, y_train_imdb = get_dataset('train')
X_test_imdb, y_test_imdb = get_dataset('test')
print(f"Train size: {len(X_train_imdb)}, Test size: {len(X_test_imdb)}")

Train size: 25000, Test size: 25000


In [15]:
def tokenize_imdb(text):
    return text.split()

tokenized_train = [tokenize_imdb(t) for t in X_train_imdb]
tokenized_test  = [tokenize_imdb(t) for t in X_test_imdb]

counter = Counter()
for toks in tokenized_train:
    counter.update(toks)

PAD_IDX = 0
UNK_IDX = 1
vocab_imdb = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
max_vocab_size = 20000
for word, _ in counter.most_common(max_vocab_size - 2):
    vocab_imdb[word] = len(vocab_imdb)

vocab_size_imdb = len(vocab_imdb)
print("Размер словаря:", vocab_size_imdb)

def numericalize(tokens):
    return [vocab_imdb.get(t, UNK_IDX) for t in tokens]

numericalized_train = [numericalize(toks) for toks in tokenized_train]
numericalized_test  = [numericalize(toks) for toks in tokenized_test]

Размер словаря: 20000


In [16]:
lengths_train = [len(toks) for toks in tokenized_train]
MAX_LEN_IMDB = int(np.percentile(lengths_train, 95))
print(f"Максимальная длина: {MAX_LEN_IMDB}")

def pad_seq(seq, max_len):
    if len(seq) < max_len:
        return seq + [PAD_IDX] * (max_len - len(seq))
    else:
        return seq[:max_len]

X_train_pad = [pad_seq(seq, MAX_LEN_IMDB) for seq in numericalized_train]
X_test_pad  = [pad_seq(seq, MAX_LEN_IMDB) for seq in numericalized_test]

X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
X_test_t  = torch.tensor(X_test_pad, dtype=torch.long)
y_train_t = torch.tensor(y_train_imdb, dtype=torch.long)
y_test_t  = torch.tensor(y_test_imdb, dtype=torch.long)

print("Размеры:", X_train_t.shape, X_test_t.shape)

Максимальная длина: 598
Размеры: torch.Size([25000, 598]) torch.Size([25000, 598])


In [17]:
X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    X_train_t, y_train_t, test_size=0.2, random_state=42, stratify=y_train_t
)

batch_size_imdb = 64
train_dataset = TextDataset(X_train_sub, y_train_sub)
val_dataset   = TextDataset(X_val_sub, y_val_sub)
test_dataset  = TextDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size_imdb, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size_imdb, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size_imdb, shuffle=False)

In [18]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 num_classes, pad_idx=0, dropout_p=0.5, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_p if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout_p)
        fc_in_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_in_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (h_n, c_n) = self.lstm(emb)
        if self.bidirectional:
            last_hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            last_hidden = h_n[-1]
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden)
        return logits

In [19]:
embed_dims = [100]
hidden_dims = [128]
dropouts = [0.3, 0.5]
weight_decays = [1e-4, 1e-5]
num_layers = 2
bidirectional = True

best_val_acc = 0.0
best_params_imdb = {}

for embed_dim in embed_dims:
    for hidden_dim in hidden_dims:
        for dropout_p in dropouts:
            for weight_decay in weight_decays:
                print(f"\nTesting embed_dim={embed_dim}, hidden_dim={hidden_dim}, dropout={dropout_p}, wd={weight_decay}")
                model = LSTMClassifier(
                    vocab_size=vocab_size_imdb,
                    embed_dim=embed_dim,
                    hidden_dim=hidden_dim,
                    num_layers=num_layers,
                    num_classes=2,
                    pad_idx=PAD_IDX,
                    dropout_p=dropout_p,
                    bidirectional=bidirectional
                ).to(device)

                optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
                criterion = nn.CrossEntropyLoss()

                patience = 2
                best_val = 0.0
                no_improve = 0
                for epoch in range(1, 4):
                    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
                    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
                    print(f"Epoch {epoch}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")
                    if val_acc > best_val:
                        best_val = val_acc
                        no_improve = 0
                    else:
                        no_improve += 1
                        if no_improve >= patience:
                            break
                print(f"Val accuracy: {best_val:.3f}")
                if best_val > best_val_acc:
                    best_val_acc = best_val
                    best_params_imdb = {
                        'embed_dim': embed_dim,
                        'hidden_dim': hidden_dim,
                        'dropout_p': dropout_p,
                        'weight_decay': weight_decay
                    }

print("\nЛучшие параметры для IMDB:", best_params_imdb)
print("Лучшая точность на валидации:", best_val_acc)


Testing embed_dim=100, hidden_dim=128, dropout=0.3, wd=0.0001
Epoch 1: train_acc=0.587, val_acc=0.641
Epoch 2: train_acc=0.657, val_acc=0.633
Epoch 3: train_acc=0.672, val_acc=0.651
Val accuracy: 0.651

Testing embed_dim=100, hidden_dim=128, dropout=0.3, wd=1e-05
Epoch 1: train_acc=0.602, val_acc=0.655
Epoch 2: train_acc=0.605, val_acc=0.657
Epoch 3: train_acc=0.696, val_acc=0.678
Val accuracy: 0.678

Testing embed_dim=100, hidden_dim=128, dropout=0.5, wd=0.0001
Epoch 1: train_acc=0.571, val_acc=0.613
Epoch 2: train_acc=0.597, val_acc=0.622
Epoch 3: train_acc=0.579, val_acc=0.576
Val accuracy: 0.622

Testing embed_dim=100, hidden_dim=128, dropout=0.5, wd=1e-05
Epoch 1: train_acc=0.576, val_acc=0.500
Epoch 2: train_acc=0.518, val_acc=0.561
Epoch 3: train_acc=0.607, val_acc=0.639
Val accuracy: 0.639

Лучшие параметры для IMDB: {'embed_dim': 100, 'hidden_dim': 128, 'dropout_p': 0.3, 'weight_decay': 1e-05}
Лучшая точность на валидации: 0.6778


In [20]:
X_full = torch.cat([X_train_sub, X_val_sub], dim=0)
y_full = torch.cat([y_train_sub, y_val_sub], dim=0)
full_dataset = TextDataset(X_full, y_full)
full_loader = DataLoader(full_dataset, batch_size=batch_size_imdb, shuffle=True)

final_model = LSTMClassifier(
    vocab_size=vocab_size_imdb,
    embed_dim=best_params_imdb['embed_dim'],
    hidden_dim=best_params_imdb['hidden_dim'],
    num_layers=num_layers,
    num_classes=2,
    pad_idx=PAD_IDX,
    dropout_p=best_params_imdb['dropout_p'],
    bidirectional=bidirectional
).to(device)

optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=best_params_imdb['weight_decay'])
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 11):
    train_loss, train_acc = train_epoch(final_model, full_loader, optimizer, criterion, device)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f}")

test_loss, test_acc = eval_epoch(final_model, test_loader, criterion, device)
print(f"\nTest accuracy: {test_acc:.3f}")

torch.save(final_model.state_dict(), 'imdb_best_model.pt')
print("Модель сохранена как 'imdb_best_model.pt'")

Epoch 01 | train_loss=0.6632 train_acc=0.598
Epoch 02 | train_loss=0.6276 train_acc=0.644
Epoch 03 | train_loss=0.5888 train_acc=0.682
Epoch 04 | train_loss=0.4698 train_acc=0.782
Epoch 05 | train_loss=0.3229 train_acc=0.866
Epoch 06 | train_loss=0.2411 train_acc=0.908
Epoch 07 | train_loss=0.1861 train_acc=0.931
Epoch 08 | train_loss=0.1320 train_acc=0.954
Epoch 09 | train_loss=0.0988 train_acc=0.967
Epoch 10 | train_loss=0.0717 train_acc=0.978

Test accuracy: 0.865
Модель сохранена как 'imdb_best_model.pt'
